## http://www.stateconstitutions.umd.edu/texts/

Important links

https://medium.com/@abdelfatahmennoun4/scraping-and-downloading-resources-from-a-web-page-using-python-3b1e30d28e9b

In [ ]:
import requests
from bs4 import BeautifulSoup, SoupStrainer
import os
import shutil
from google.colab import files as colab_files # Renamed to avoid conflict later

constitutions_url = 'http://www.stateconstitutions.umd.edu/texts/'
base_url = 'http://www.stateconstitutions.umd.edu'
save_folder = 'constitutions'

# Create the folder if it doesn't exist
if not os.path.exists(save_folder):
    os.makedirs(save_folder)

# Headers to mimic a real browser (prevents some 403 Forbidden errors)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Fetch the page content
try:
    response = requests.get(constitutions_url, headers=headers)
    response.raise_for_status() # Check for connection errors
    content = response.text
except Exception as e:
    print(f"Failed to connect to {constitutions_url}: {e}")
    # Stop execution if we can't get the page
    raise SystemExit

# Parse HTML to find links
# Renamed 'files' list to 'file_links' to avoid overwriting the 'files' module
file_links = []
for link in BeautifulSoup(content, "lxml", parse_only=SoupStrainer('a')):
    if link.has_attr('href'):
        href = link['href']
        # Filter: Only include links that are substantial (length > 1)
        if len(href) > 1:
            # We only care about links inside the /texts/ folder
            if href.startswith('/texts/'):
                file_links.append(href)

print(f"Found {len(file_links)} files to download.")

# Download Loop
rejects = []
rejectNames = []

for file_path in file_links:
    # Construct the full URL
    full_url = base_url + file_path

    # Construct the filename (e.g., '/texts/AK.txt' -> 'AK.txt')
    filename = file_path.replace('/texts/', '')
    save_path = os.path.join(save_folder, filename)

    # Check if file already exists
    if not os.path.isfile(save_path):
        try:
            print(f"Downloading: {filename}...")
            file_response = requests.get(full_url, headers=headers)

            if file_response.status_code == 200:
                # Write content to file using utf-8 encoding (standard for text)
                with open(save_path, 'w', encoding='utf-8', errors='ignore') as f:
                    f.write(file_response.text)
            else:
                raise Exception(f"Status code {file_response.status_code}")

        except Exception as e:
            print(f"Failed to download {filename}: {e}")
            rejects.append(full_url)
            rejectNames.append(filename)
    else:
        print(f"Skipping {filename} (already exists)")

# Retry failed downloads (optional, mimicking original logic)
if rejects:
    print("\nRetrying failed downloads...")
    for reject_url in rejects:
        # Extract filename from reject URL
        # logic: http://www.stateconstitutions.umd.edu/texts/AK.txt -> AK.txt
        filename = reject_url.replace(base_url + '/texts/', '')
        save_path = os.path.join(save_folder, filename)

        if not os.path.isfile(save_path):
            try:
                print(f"Retrying: {filename}...")
                r = requests.get(reject_url, headers=headers)
                with open(save_path, 'w', encoding='utf-8', errors='ignore') as code:
                    code.write(r.text)
            except Exception as e:
                print(f"Retry failed for {filename}: {e}")

# Zip and Download for Google Colab
print("\nZipping files...")
zip_filename = 'state_constitutions_archive'
# shutil.make_archive creates the zip file
shutil.make_archive(zip_filename, 'zip', save_folder)

print("Downloading zip file...")
# Using the renamed module 'colab_files' to trigger the download
colab_files.download(zip_filename + ".zip")

Found 144 files to download.
Downloading: AK1836_final_parts_0.txt...
Downloading: AK1959_final_parts_0.txt...
Downloading: AL1819_final_parts_0.txt...
Downloading: AL1861_final_parts_0.txt...
Downloading: AL1865_final_parts_0.txt...
Downloading: AL1868_final_parts_0.txt...
Downloading: AL1875_final_parts_0.txt...
Downloading: AL1901_001200_final_parts_0.txt...
Downloading: AL1901_201400_final_parts_0.txt...
Downloading: AL1901_401658_final_parts_0.txt...
Downloading: AL1901_final_parts_0.txt...
Downloading: AR1864_final_parts_0.txt...
Downloading: AR1868_final_parts_0.txt...
Downloading: AR1874_final_parts_0.txt...
Downloading: AZ1912_final_parts_0.txt...
Downloading: CO1876_amd_final_parts.txt...
Downloading: CO1876_amd_final_parts_0.txt...
Downloading: CO1876_final_parts_0.txt...
Downloading: CT1662_final_parts_0.txt...
Downloading: CT1818_final_parts_0.txt...
Downloading: CT1955_final_parts_0.txt...
Downloading: CT1965_final_parts_0.txt...
Downloading: DE1776_final_parts_0.txt...
D

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>